# Lab 9 · LangGraph: parallel workers and explicit state
## Goal
Build a bounded equipment-policy review with `StateGraph`, dynamic `Send` workers, a reducer, and an aggregation barrier. Explain why collecting worker results is different from trusting them. Prerequisites: Labs 6 and 8, Python types and dictionaries; textbook chapters 4, 7, 8, and 11.

Estimated 110 minutes: 15 reading, 20 worked trace, 45 implementation, 20 failure cases, 10 explanation. By the end you will handle zero workers, preserve source identity, bound fan-out, and make deterministic output independent of worker completion order.

This exercise focuses on orchestration. Workers perform a transparent lexical check rather than pretend to be intelligent reviewers. A later model substitution must be evaluated separately.

## Setup
Use the separate `labs/frameworks/requirements.txt` environment described in this track's README. Do not upgrade the earlier lab environment in place. Each notebook is self-contained after installation. Python 3.12 is the tested interpreter family.

Default execution uses invented equipment-service fixtures and real framework runtimes, without network inference. These checks establish application behavior, not model accuracy. Student tasks are disabled until you implement them and set `RUN_EXERCISES=True`. Instructor copies enable them. Optional live cells require explicit `RUN_LIVE=True`, `GEMINI_API_KEY`, and `GEMINI_MODEL`. Choose a model available to your account; API use consumes quota. Keep credentials out of notebook cells and saved output.

In [1]:
import os
from importlib.metadata import version
RUN_EXERCISES = False
RUN_LIVE = False

def live_config():
    key, model = os.getenv("GEMINI_API_KEY"), os.getenv("GEMINI_MODEL")
    if not key or not model:
        raise RuntimeError("Set GEMINI_API_KEY and GEMINI_MODEL before enabling live calls")
    return key, model

print({p: version(p) for p in ("langgraph", "langchain", "pydantic-ai-slim")})

{'langgraph': '1.2.14', 'langchain': '1.4.3', 'pydantic-ai-slim': '2.54.0'}


## Steps
### 1. Define the workload and its contract
A document has a unique, nonempty string ID and nonempty text. Accept at most eight documents. Reject duplicates rather than letting a reducer silently overwrite one. The batch validator runs before any worker is scheduled. This is a teaching limit, not a framework limit.

In [2]:
from typing import Annotated, TypedDict
import operator
from langgraph.graph import StateGraph, START, END
from langgraph.types import Send

class ReviewState(TypedDict):
    docs: list[dict]
    findings: Annotated[list[dict], operator.add]
    summary: list[dict]

class WorkerState(TypedDict):
    doc: dict

def validate_docs(docs):
    if not isinstance(docs, list) or len(docs) > 8:
        raise ValueError("docs must be a list of at most eight documents")
    seen = set()
    for doc in docs:
        if not isinstance(doc, dict) or set(doc) != {"id", "text"}:
            raise ValueError("document requires exactly id and text")
        if any(not isinstance(doc[k], str) or not doc[k].strip() for k in ("id", "text")):
            raise ValueError("id and text must be nonempty strings")
        if doc["id"] in seen:
            raise ValueError("duplicate document ID")
        seen.add(doc["id"])
    return docs

def worker(state: WorkerState):
    doc = state["doc"]
    return {"findings": [{"id": doc["id"], "mentions_approval": "approval" in doc["text"].lower()}]}

def route(state):
    return [Send("review", {"doc": doc}) for doc in state["docs"]] or "aggregate"

def aggregate(state):
    return {"summary": sorted(state["findings"], key=lambda row: row["id"])}

def make_graph(worker_fn=worker):
    graph = StateGraph(ReviewState)
    graph.add_node("validate", lambda state: {"docs": validate_docs(state["docs"])})
    graph.add_node("review", worker_fn)
    graph.add_node("aggregate", aggregate)
    graph.add_edge(START, "validate")
    graph.add_conditional_edges("validate", route, ["review", "aggregate"])
    graph.add_edge("review", "aggregate")
    graph.add_edge("aggregate", END)
    return graph.compile()

app = make_graph()
docs = [{"id":"B", "text":"Approval required for loans."}, {"id":"A", "text":"Bring student ID."}]
result = app.invoke({"docs": docs, "findings": [], "summary": []}, config={"max_concurrency":2})
assert result["summary"] == [{"id":"A", "mentions_approval":False}, {"id":"B", "mentions_approval":True}]
print(result["summary"])

[{'id': 'A', 'mentions_approval': False}, {'id': 'B', 'mentions_approval': True}]


### 2. Trace the graph
`Send` gives each worker its own input. The reducer appends updates into the shared findings list. Sorting is an explicit presentation decision; do not depend on worker finish order. This graph does not have a checkpointer: invoke starts a fresh run. An in-memory result is not durable storage.

The validator costs O(n) plus string inspection; sorting n findings costs O(n log n), with O(n) stored results. Concurrency can overlap work but does not prove a speedup for these tiny Python functions. Total provider calls would grow with worker count if inference were added.

In [3]:
updates = list(app.stream({"docs":docs, "findings":[], "summary":[]}, stream_mode="updates"))
print([list(event) for event in updates])
assert app.invoke({"docs":[], "findings":[], "summary":[]})["summary"] == []
for bad in (None, {}, [docs[0], docs[0]], [{"id":"", "text":"x"}], [{"id":"x","text":1}], docs * 5):
    try:
        app.invoke({"docs":bad, "findings":[], "summary":[]})
    except ValueError:
        pass
    else:
        raise AssertionError("invalid batch accepted")
print("Worked boundary checks passed")

[['validate'], ['review'], ['review'], ['aggregate']]
Worked boundary checks passed


### 3. TODO: build a conservative consensus reducer (35 points)
Implement `merge_findings(rows)` returning one row per ID in sorted order. A row must contain exactly `id` and a boolean `mentions_approval`. Reject malformed values. Repeated identical findings collapse; conflicting values for an ID raise `ValueError`. Do not resolve a disagreement by last-write-wins. Empty input returns an empty list.

In [4]:
def merge_findings(rows):
    raise NotImplementedError("TODO: validate, deduplicate, and detect disagreement")

In [5]:
if RUN_EXERCISES:
    assert merge_findings([]) == []
    assert merge_findings(result["summary"] * 2) == result["summary"]
    for bad in (None, [{"id":"A", "mentions_approval":1}], [{"id":"A", "mentions_approval":True}, {"id":"A", "mentions_approval":False}]):
        try: merge_findings(bad)
        except ValueError: pass
        else: raise AssertionError("bad findings accepted")

### 4. TODO: integrate the reducer into the graph (35 points)
Build `make_consensus_graph(worker_fn)` with the same validation, routing, worker, and aggregation topology. Aggregation must call your `merge_findings`, not the worked `aggregate`. Demonstrate an injected worker returning duplicates and another returning a conflict. Dependency injection lets tests change behavior without network inference.

In [6]:
def make_consensus_graph(worker_fn):
    raise NotImplementedError("TODO: wire the graph to merge_findings")

In [7]:
if RUN_EXERCISES:
    def doubled(s): return {"findings":worker(s)["findings"] * 2}
    assert make_consensus_graph(doubled).invoke({"docs":docs,"findings":[],"summary":[]})["summary"] == result["summary"]
    def conflict(s):
        key=s["doc"]["id"]
        return {"findings":[{"id":key,"mentions_approval":True},{"id":key,"mentions_approval":False}]}
    try: make_consensus_graph(conflict).invoke({"docs":docs[:1],"findings":[],"summary":[]})
    except ValueError: pass
    else: raise AssertionError("conflict hidden")
    assert make_consensus_graph(worker).invoke({"docs":[],"findings":[],"summary":[]})["summary"] == []
    print("Student graph checks passed")

## Checks and submission
Submit the completed notebook and a short trace explaining validation → fan-out → aggregation. Add a one-document test, an eight-document test, and a test showing invalid input schedules no worker. Explain why the lexical rule fails on “approval is not required.” Replace it only after defining a labeled evaluation set.

Rubric: conservative merge 35; graph integration 35; added boundary tests 15; trace and limitations 15. The included checks are a starter suite, not exhaustive grading.

## Next Steps
Use Lab 8 for durable approval; combine it with this graph only after deciding what state and effects can be replayed. Compare the explicit graph to Lab 10's agent loop and Lab 7's Agents SDK runner. Framework choice is a design tradeoff, not a benchmark ranking.

References checked 2026-10-06: [LangGraph Graph API: Send, reducers, edges](https://docs.langchain.com/oss/python/langgraph/graph-api), [persistence](https://docs.langchain.com/oss/python/langgraph/persistence). The implementation and tests are original course examples.